In [ ]:
# Imports
import os
import re
import cv2
import gc
import glob
import shutil
import requests
import numpy as np
import matplotlib.pyplot as plt
from io import BytesIO
from PIL import Image, ImageDraw
from tqdm import tqdm
from google.colab import drive

# Montage du Google Drive
drive.mount('/content/drive')

# Chemins globaux
PROJECT_NAME = "Reconstruction"
CHECKPOINT_DIR = "/content/drive/MyDrive/checkpoints_reconstruction"
DATASET_LOCAL_DIR = "/content/dataset"

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

In [ ]:
# Accès Git et installation des dépendances
%cd /content/
if not os.path.exists('pytorch-CycleGAN-and-pix2pix'):
    !git clone https://github.com/junyanz/pytorch-CycleGAN-and-pix2pix

# Installation des paquets requis par le dépôt officiel
!pip install -q dominate visdom wandb scipy tqdm opencv-python

In [ ]:
import os
import shutil
import requests
from tqdm import tqdm
from io import BytesIO
from PIL import Image, ImageDraw

# Création du dataset

import random

def couper_un_morceau(image_pil):
    """Génère un carré noir à un emplacement ALÉATOIRE."""
    img_trouee = image_pil.copy()
    draw = ImageDraw.Draw(img_trouee)

    taille = 96 # Taille du carré (176 - 80)

    # Coordonnées aléatoires pour que le carré reste dans l'image 256x256
    x1 = random.randint(10, 256 - taille - 10)
    y1 = random.randint(10, 256 - taille - 10)
    x2 = x1 + taille
    y2 = y1 + taille

    draw.rectangle([x1, y1, x2, y2], fill="black")
    return img_trouee

def combiner_A_et_B(image_A, image_B):
    """Combine l'image abîmée (A) et originale (B) côte à côte (512x256)."""
    largeur, hauteur = image_B.size
    image_combinee = Image.new("RGB", (largeur * 2, hauteur))
    image_combinee.paste(image_A, (0, 0))
    image_combinee.paste(image_B, (largeur, 0))
    return image_combinee

# Nettoyage des anciens dossiers
if os.path.exists(DATASET_LOCAL_DIR):
    shutil.rmtree(DATASET_LOCAL_DIR)

os.makedirs(os.path.join(DATASET_LOCAL_DIR, "train"), exist_ok=True)
os.makedirs(os.path.join(DATASET_LOCAL_DIR, "val"), exist_ok=True)

# Téléchargement des données d'entraînement (1000 paires)
img_train_telechargees = 0
pbar_train = tqdm(total=1000, desc="Téléchargement Train")

while img_train_telechargees < 1000:
    try:
        # Chaque index produit une image unique et reproductible
        url = f"https://picsum.photos/seed/reconstruct_train_{img_train_telechargees}/256/256"
        response = requests.get(url, timeout=5)

        if response.status_code == 200:
            img_originale = Image.open(BytesIO(response.content)).convert('RGB')
            img_abimee = couper_un_morceau(img_originale)
            img_ready_for_pix2pix = combiner_A_et_B(img_abimee, img_originale)

            nom_fichier = f"{DATASET_LOCAL_DIR}/train/image_{img_train_telechargees}.jpg"
            img_ready_for_pix2pix.save(nom_fichier, "JPEG")

            img_train_telechargees += 1
            pbar_train.update(1)
    except Exception:
        continue
pbar_train.close()
# Téléchargement de la Validation (100 paires) ---
img_val_telechargees = 0
pbar_val = tqdm(total=100, desc="Téléchargement Val")

while img_val_telechargees < 100:
    try:
        # Utilisation d'un préfixe distinct pour garantir des images 100% différentes du Train
        url = f"https://picsum.photos/seed/reconstruct_val_{img_val_telechargees}/256/256"
        response = requests.get(url, timeout=5)

        if response.status_code == 200:
            img_originale = Image.open(BytesIO(response.content)).convert('RGB')
            img_abimee = couper_un_morceau(img_originale)
            img_ready_for_pix2pix = combiner_A_et_B(img_abimee, img_originale)

            nom_fichier = f"{DATASET_LOCAL_DIR}/val/image_val_{img_val_telechargees}.jpg"
            img_ready_for_pix2pix.save(nom_fichier, "JPEG")

            img_val_telechargees += 1
            pbar_val.update(1)
    except Exception:
        continue
pbar_val.close()

# Vérification finale
nb_train = len(os.listdir(os.path.join(DATASET_LOCAL_DIR, "train")))
nb_val = len(os.listdir(os.path.join(DATASET_LOCAL_DIR, "val")))

In [ ]:
# Checkpoint et entraînement
checkpoint_project_dir = os.path.join(CHECKPOINT_DIR, PROJECT_NAME)
last_epoch = 0
continue_flag = ""

if os.path.exists(checkpoint_project_dir):
    files = os.listdir(checkpoint_project_dir)
    epochs = [int(re.findall(r'(\d+)_net_G.pth', f)[0]) for f in files if 'net_G.pth' in f and 'latest' not in f]
    if epochs:
        last_epoch = max(epochs)
        continue_flag = "--continue_train"
        print(f"Dernier checkpoint : Epoch {last_epoch}")

resume_epoch = last_epoch + 1

if nb_train > 0:
    print(f"Lancement de l'entraînement pour la reconstruction : Epoch {resume_epoch}...")
    %cd /content/pytorch-CycleGAN-and-pix2pix

    !python train.py --dataroot {DATASET_LOCAL_DIR} \
                     --name {PROJECT_NAME} \
                     --model pix2pix \
                     --direction AtoB \
                     --dataset_mode aligned \
                     --input_nc 3 \
                     --output_nc 3 \
                     --n_epochs 200 \
                     --n_epochs_decay 200 \
                     --batch_size 8 \
                     --checkpoints_dir {CHECKPOINT_DIR} \
                     --save_epoch_freq 10 \
                     --lambda_L1 100 \
                     {continue_flag} \
                     --gan_mode lsgan
else:
    print("L'entraînement ne peut pas démarrer.")

In [ ]:
#Script de validation
import os
import glob
import torch
import numpy as np
import cv2
import matplotlib.pyplot as plt

%cd /content/pytorch-CycleGAN-and-pix2pix
from models.networks import define_G

# Mettre à jour avec le bon chemin de vos poids d'Inpainting
PATH_TO_WEIGHTS = "/content/drive/MyDrive/checkpoints_reconstruction/Reconstruction/last_net_G.pth"
VAL_IMAGES_DIR = "/content/dataset/val/"

if not os.path.exists(PATH_TO_WEIGHTS):
    print(f"Impossible de trouver les poids du modèle à l'adresse : {PATH_TO_WEIGHTS}")
else:
    print("Chargement du générateur Pix2Pix pour la Reconstruction (Unet 256, 3 in / 3 out)...")
    # Modification fondamentale : input_nc=3 et output_nc=3
    netG = define_G(input_nc=3, output_nc=3, ngf=64, netG='unet_256', norm='batch', use_dropout=False, init_type='normal', init_gain=0.02)

    state_dict = torch.load(PATH_TO_WEIGHTS, map_location='cuda:0')
    if hasattr(state_dict, '_metadata'): del state_dict._metadata
    netG.load_state_dict(state_dict)
    netG.eval()
    netG.cuda()

    # 1. Récupération des images du dataset de validation aligné (images de 512x256)
    images_dispos = sorted(glob.glob(os.path.join(VAL_IMAGES_DIR, "*.jpg")))

    if len(images_dispos) == 0:
        print(f"Aucune image trouvée dans {VAL_IMAGES_DIR}")
    else:
        nb_images_a_afficher = min(100, len(images_dispos))
        print(f"Génération de la galerie de reconstruction pour {nb_images_a_afficher} images...\n")

        for i, img_path in enumerate(images_dispos[:nb_images_a_afficher]):
            img_bgr = cv2.imread(img_path)
            if img_bgr is None: continue

            # Conversion en RGB
            img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

            # Découpe de l'image alignée (largeur 512 divisée en deux blocs de 256)
            # Partie Gauche : Image abîmée (A) | Partie Droite : Image originale cible (B)
            img_abimee = img_rgb[:, :256, :]
            img_cible = img_rgb[:, 256:, :]

            # Normalisation entre [-1, 1] requise par Pix2Pix : (img / 127.5) - 1.0
            img_input = (img_abimee / 127.5) - 1.0

            # Transposition du format (H, W, C) vers le format PyTorch (C, H, W)
            img_input_t = img_input.transpose(2, 0, 1)

            # Ajout de la dimension Batch et envoi sur le GPU
            tens_input = torch.from_numpy(img_input_t).float().unsqueeze(0).cuda()

            # --- Inférence du modèle ---
            with torch.no_grad():
                output_norm = netG(tens_input)

            # --- Post-traitement de la sortie ---
            # Suppression du batch, rapatriement sur CPU et retour au format numpy (H, W, C)
            output_np = output_norm.squeeze(0).cpu().float().numpy().transpose(1, 2, 0)

            # Dénormalisation de [-1, 1] vers [0, 255]
            rgb_reconstruit = ((output_np + 1.0) * 127.5).clip(0, 255).astype(np.uint8)

            # --- Affichage de la ligne de comparaison ---
            fig, axes = plt.subplots(1, 3, figsize=(15, 4))

            # Colonne 1 : L'entrée fournie au réseau (l'image trouée)
            axes[0].imshow(img_abimee)
            axes[0].set_title(f"Image {i+1} : Entrée Abîmée (Trou noir)")
            axes[0].axis('off')

            # Colonne 2 : La reconstruction magique faite par le réseau
            axes[1].imshow(rgb_reconstruit)
            axes[1].set_title("Reconstruction Pix2Pix")
            axes[1].axis('off')

            # Colonne 3 : La vérité terrain d'origine
            axes[2].imshow(img_cible)
            axes[2].set_title("Cible Attendue (Ground Truth)")
            axes[2].axis('off')

            plt.show()